# Run CLARA Shiny app in Google Colab

This notebook keeps all installation and files inside the Colab VM (no changes on your local machine). Follow the cells in order. Use Drive or upload files; the app will run inside Colab and be visible only inside the notebook session.

**Quick notes**

- This runs entirely inside the Colab virtual machine.
- You can either mount Google Drive (recommended) or upload files directly.
- The Shiny app will run on port 3838 inside Colab and be embedded in an iframe.
- Colab sessions are ephemeral; save any outputs to Drive.

In [ ]:
# Optional: mount Google Drive and copy your project 'main' folder into /content/main
from google.colab import drive
drive.mount('/content/drive')

# After mounting, copy the CLARA 'main' folder from your Drive to /content (adjust path):
# !cp -r "/content/drive/My Drive/path/to/CLARA/main" /content/main
# Verify files:
# !ls -la /content/main

In [ ]:
%%bash
set -e
# Install system packages and R inside the Colab VM (only affects the VM)
apt-get update -y
apt-get install -y --no-install-recommends wget unzip jq ca-certificates
DEBIAN_FRONTEND=noninteractive apt-get install -y --no-install-recommends r-base
echo 'System packages and R installed.'

In [ ]:
%%bash
set -e
# Install CRAN packages used by CLARA inside the Colab R environment
Rscript -e "options(repos='https://cran.rstudio.com/'); pkgs <- c('shiny','shinyjs','readxl','DT','ggplot2','dplyr','jsonlite','ggtext','shinyWidgets','digest','tibble','tidyr','ggpattern','emmeans','multcomp','multcompView','sortable','commonmark','fBasics','afex','rstatix','dunn.test'); install.packages(pkgs, dependencies=TRUE)"
echo 'R packages installation attempted (may take several minutes).'

If you did not use Drive, upload the project files now (the files must be placed into `/content/main`).

In [ ]:
# Upload files (use this if you prefer direct upload instead of Drive)
from google.colab import files
uploaded = files.upload()
# After upload, move files into /content/main. Example if you uploaded a zip or individual .R files:
# import os
# os.makedirs('/content/main', exist_ok=True)
# !mv *.R /content/main/

In [ ]:
%%bash
set -e
# Ensure /content/main exists and contains the app files (05_app.R etc.).
if [ ! -d /content/main ]; then
  echo '/content/main not found. Please copy your CLARA main folder here (see previous cells).'
  exit 1
fi
# Start the Shiny app in background on 0.0.0.0:3838 (inside Colab VM). Logs go to shiny.log
nohup Rscript -e "shiny::runApp('main', host='0.0.0.0', port=3838)" > shiny.log 2>&1 &
sleep 1
echo 'Shiny started (check shiny.log for output).'

In [ ]:
# Embed the app inside the notebook using Colab iframe serving (visible inside notebook).
from google.colab import output
output.serve_kernel_port_as_iframe(3838, height=800)

**Tips & troubleshooting**

- Save any generated files back to Drive if you need persistence.
- If package installation fails, re-run the R install cell; installing packages inside Colab VM does not affect your local machine.
- For public long-term deployment use `shinyapps.io` or a VPS.
- If the iframe is blank, open `shiny.log` to inspect errors: `!tail -n 200 shiny.log`.